# 15 — Le hedger comme pricer : prix d'indifférence et spread bid-ask (vanille et exotiques)

## L'idée : couvrir et coter, c'est le même problème

Un deep hedger ne fait pas que couvrir, il **cote**. Le prix vendeur (*ask*) d'une option, c'est la prime minimale telle que, *après couverture optimale*, ton risque de queue n'est pas pire que ne rien faire :

$$p_{\text{ask}} = \inf\{p : \rho(\text{position vendue et couverte au prix } p) \le 0\}.$$

Avec $\rho = \text{CVaR}_\alpha$ et l'invariance par translation ($\text{CVaR}(L - c) = \text{CVaR}(L) - c$), ça se résout tout seul :

$$p_{\text{ask}} = e^{-rT}\,\underbrace{\min_{\text{couverture}} \text{CVaR}_\alpha\big(\text{payoff} - \text{gains de couverture}\big)}_{a},\qquad p_{\text{bid}} = -\,e^{-rT}\,\underbrace{\min_{\text{couverture}} \text{CVaR}_\alpha\big(-\text{gains} - \text{payoff}\big)}_{b}.$$

Le terme à minimiser, c'est **exactement ce que le réseau optimise** (à prime nulle). D'où :

$$\boxed{\text{spread} = p_{\text{ask}} - p_{\text{bid}} = e^{-rT}(a + b)}$$

où $a$ et $b$ sont les risques résiduels de couverture des positions short et long. En marché complet et sans coûts, $a = b = 0$ : réplication parfaite, $p_{\text{ask}} = p_{\text{bid}} = $ prix risque-neutre, **pas de spread**. Dès qu'il y a des frictions ou de l'incomplétude, $a, b > 0$ : le spread émerge, et sa largeur mesure la **difficulté de couverture du produit**. C'est littéralement comme ça qu'un market maker cote.

On le montre sur trois produits de difficulté croissante, sous GBM (pour isoler l'effet du produit) avec coût 1% : un **call vanille**, une **barrière up-and-out**, une **digitale** (paie 1 si $S_T \ge K$).


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.stats import norm

torch.manual_seed(0)
S0, K, r, T, sigma = 100., 100., 0.02, 1.0, 0.20
n, cost, alpha = 63, 0.01, 0.95; dt = T/n; Bar = 130.0

# --- prix risque-neutre fermes (references) ---
def d12(S, tau): d1=(np.log(S/K)+(r+0.5*sigma**2)*tau)/(sigma*np.sqrt(tau)); return d1, d1-sigma*np.sqrt(tau)
def bs_call(S, tau): d1,d2=d12(S,tau); return S*norm.cdf(d1)-K*np.exp(-r*tau)*norm.cdf(d2)
def bs_call_delta(S, tau): d1,_=d12(S,tau); return norm.cdf(d1)
def dig_price(S, tau): _,d2=d12(S,tau); return np.exp(-r*tau)*norm.cdf(d2)
def dig_delta(S, tau): _,d2=d12(S,tau); return np.exp(-r*tau)*norm.pdf(d2)/(S*sigma*np.sqrt(tau))
def uo_call(S, tau, sig=sigma):
    S=np.asarray(S,float); H=Bar; srt=sig*np.sqrt(tau); m_=(r-0.5*sig**2)/sig**2
    x1=np.log(S/K)/srt+(1+m_)*srt; x2=np.log(S/H)/srt+(1+m_)*srt
    y1=np.log(H**2/(S*K))/srt+(1+m_)*srt; y2=np.log(H/S)/srt+(1+m_)*srt
    A=S*norm.cdf(x1)-K*np.exp(-r*tau)*norm.cdf(x1-srt); B=S*norm.cdf(x2)-K*np.exp(-r*tau)*norm.cdf(x2-srt)
    C=S*(H/S)**(2*(m_+1))*norm.cdf(-y1)-K*np.exp(-r*tau)*(H/S)**(2*m_)*norm.cdf(-y1+srt)
    D=S*(H/S)**(2*(m_+1))*norm.cdf(-y2)-K*np.exp(-r*tau)*(H/S)**(2*m_)*norm.cdf(-y2+srt)
    return np.where(S>=H,0.,A-B+C-D)
def uo_delta(S, tau, h=0.5): return (uo_call(S+h,tau)-uo_call(S-h,tau))/(2*h)
def CVaR(loss, a=0.95): return loss[loss>=np.quantile(loss,a)].mean()

RN = {'call': float(bs_call(S0,T)), 'barriere': float(uo_call(S0,T)), 'digitale': float(dig_price(S0,T))}
print("prix risque-neutre :", {k: round(v,3) for k,v in RN.items()})


## Référence classique : spread du delta-hedge

On calcule d'abord le spread qu'un hedger classique (delta du produit, rééquilibré à chaque pas) produirait. Ça montre déjà que le spread explose sur les exotiques, et ça servira de point de comparaison au réseau.


In [ ]:
rng = np.random.default_rng(0); times = np.linspace(0,T,n+1)
def sim_np(m):
    Z=rng.standard_normal((m,n)); inc=(r-0.5*sigma**2)*dt+sigma*np.sqrt(dt)*Z
    return S0*np.exp(np.concatenate([np.zeros((m,1)),np.cumsum(inc,axis=1)],axis=1))
Sc = sim_np(200_000); alive_c = np.cumprod((Sc<Bar).astype(float),axis=1)
prod = {
 'call':     dict(payoff=np.maximum(Sc[:,-1]-K,0.),                      delta=lambda s,t: bs_call_delta(s,t), gate=None),
 'barriere': dict(payoff=np.where(alive_c[:,-1]>0,np.maximum(Sc[:,-1]-K,0.),0.), delta=lambda s,t: uo_delta(s,t), gate=alive_c),
 'digitale': dict(payoff=(Sc[:,-1]>=K).astype(float),                    delta=lambda s,t: dig_delta(s,t),     gate=None),
}
def classic_gains(delta, sign, gate):
    m=Sc.shape[0]; cash=np.zeros(m); pos=np.zeros(m)
    for k in range(n):
        tau=max(T-times[k],1e-3); g=1.0 if gate is None else gate[:,k]
        tgt=sign*delta(Sc[:,k],tau)*g; tr=tgt-pos; cash-=tr*Sc[:,k]+cost*np.abs(tr)*Sc[:,k]; pos+=tr; cash*=np.exp(r*dt)
    return cash+pos*Sc[:,-1]
classic = {}
for name,p in prod.items():
    a=CVaR(p['payoff']-classic_gains(p['delta'],+1,p['gate']))
    b=CVaR(-classic_gains(p['delta'],-1,p['gate'])-p['payoff'])
    classic[name]=(np.exp(-r*T)*a, -np.exp(-r*T)*b)
    print(f"{name:10s} RN={RN[name]:6.3f}  ask={classic[name][0]:6.2f}  bid={classic[name][1]:6.2f}  spread={classic[name][0]-classic[name][1]:6.2f}")


## Le deep hedger : prix d'indifférence appris

Pour chaque produit, on entraîne **deux** réseaux : un qui minimise le risque de la position vendue (donne $p_{\text{ask}}$) et un pour la position achetée (donne $p_{\text{bid}}$). Simulation GBM fraîche à chaque époque, perte CVaR empirique directe. Le réseau apprend tout seul le signe de la couverture (delta positif pour répliquer un short, négatif pour un long).


In [ ]:
def sim_gbm_t(m):
    Z=torch.randn(m,n); inc=(r-0.5*sigma**2)*dt+sigma*np.sqrt(dt)*Z
    logS=torch.cat([torch.zeros(m,1), torch.cumsum(inc,dim=1)],dim=1)
    return S0*torch.exp(logS)
def cvar_torch(L,a=0.95): var=torch.quantile(L,a); return L[L>=var].mean()

class Net(torch.nn.Module):
    def __init__(self,d,h=32):
        super().__init__()
        self.net=torch.nn.Sequential(torch.nn.Linear(d,h),torch.nn.ReLU(),
                                     torch.nn.Linear(h,h),torch.nn.ReLU(),torch.nn.Linear(h,1))
    def forward(self,x): return self.net(x).squeeze(-1)

def gains_t(net, S, barrier):
    m=S.shape[0]; cash=torch.zeros(m); pos=torch.zeros(m); alive=torch.ones(m)
    for k in range(n):
        tau=float(T-k*dt)
        if barrier:
            alive=alive*(S[:,k]<Bar).float()
            feat=torch.stack([torch.log(S[:,k]/K),torch.full((m,),tau),pos,(Bar-S[:,k])/S0,alive],1)
        else:
            feat=torch.stack([torch.log(S[:,k]/K),torch.full((m,),tau),pos],1)
        d=net(feat); tr=d-pos; cash=cash-tr*S[:,k]-cost*torch.abs(tr)*S[:,k]; cash=cash*np.exp(r*dt); pos=d
    return cash+pos*S[:,-1]

def payoff_t(S, kind):
    if kind=='call':     return torch.clamp(S[:,-1]-K,min=0.)
    if kind=='digitale': return (S[:,-1]>=K).float()
    if kind=='barriere':
        alive=torch.prod((S<Bar).float(),dim=1); return torch.clamp(S[:,-1]-K,min=0.)*alive

def train_side(kind, side, epochs=300, m=20000, lr=1e-3):
    barrier=(kind=='barriere'); net=Net(5 if barrier else 3); opt=torch.optim.Adam(net.parameters(),lr)
    for ep in range(epochs):
        S=sim_gbm_t(m).detach(); g=gains_t(net,S,barrier); pay=payoff_t(S,kind)
        L = (pay-g) if side=='ask' else (-g-pay)
        loss=cvar_torch(L); opt.zero_grad(); loss.backward(); opt.step()
    return net

def price_side(net, kind, side, m=100000):
    barrier=(kind=='barriere')
    with torch.no_grad():
        S=sim_gbm_t(m); g=gains_t(net,S,barrier); pay=payoff_t(S,kind)
        L=(pay-g) if side=='ask' else (-g-pay)
        return float(np.exp(-r*T)*CVaR(L.numpy())) * (1 if side=='ask' else -1)

net_price={}
for kind in ['call','barriere','digitale']:
    print(f"entrainement {kind} ...")
    na=train_side(kind,'ask'); nb=train_side(kind,'bid')
    net_price[kind]=(price_side(na,kind,'ask'), price_side(nb,kind,'bid'))
    print(f"   ask={net_price[kind][0]:.3f}  bid={net_price[kind][1]:.3f}")


In [ ]:
print(f"{'produit':10s} {'prixRN':>7s} | {'spread classique':>16s} | {'spread reseau':>13s}")
for kind in ['call','barriere','digitale']:
    sc=classic[kind][0]-classic[kind][1]; sn=net_price[kind][0]-net_price[kind][1]
    print(f"{kind:10s} {RN[kind]:7.3f} | {sc:16.2f} | {sn:13.2f}")

kinds=['call','barriere','digitale']; x=np.arange(3); wd=0.35
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(13,4.5))
ax1.bar(x-wd/2,[(classic[k][0]-classic[k][1]) for k in kinds],wd,label='classique (delta)',color='tab:orange')
ax1.bar(x+wd/2,[(net_price[k][0]-net_price[k][1]) for k in kinds],wd,label='reseau (deep hedge)',color='tab:green')
ax1.set_xticks(x); ax1.set_xticklabels(kinds); ax1.set_ylabel('spread bid-ask'); ax1.set_title('Spread absolu'); ax1.legend()
ax2.bar(x-wd/2,[(classic[k][0]-classic[k][1])/RN[k] for k in kinds],wd,color='tab:orange')
ax2.bar(x+wd/2,[(net_price[k][0]-net_price[k][1])/RN[k] for k in kinds],wd,color='tab:green')
ax2.set_xticks(x); ax2.set_xticklabels(kinds); ax2.set_ylabel('spread / prix'); ax2.set_title('Spread relatif (difficulte de couverture)')
fig.tight_layout(); plt.show()


## Ce qu'il faut retenir

- **Le deep hedger est un pricer.** Le même réseau qui couvre produit un prix d'indifférence, et le spread bid-ask tombe directement de son risque résiduel : $\text{spread} = e^{-rT}(a+b)$. Pas de spread ad hoc, il émerge de la couverture.
- **Le spread mesure la difficulté de couverture du produit.** Serré sur le call (facile à répliquer), large sur la digitale (delta explosif au strike à l'échéance), énorme sur la barrière (delta qui change de signe, gap de désactivation). C'est le raisonnement exact d'un market maker : on cote large ce qu'on couvre mal.
- **Mieux couvrir, c'est coter plus serré.** Le réseau resserre le spread par rapport au delta classique, et l'écart est le plus grand sur les exotiques, là où le hedge classique échoue. Un desk qui couvre mieux peut coter plus serré et prendre le flux : le lien entre qualité de couverture et compétitivité commerciale est explicite.
- **Sans friction ni incomplétude, le spread s'annule** (réplication parfaite au prix risque-neutre). Le spread est donc entièrement le prix des frictions et du risque non couvrable, ce qui unifie tout l'arc du projet : le CVaR résiduel qu'on a combattu partout, c'est aussi ce qui fixe le prix.
